# 01 · Фильтр запросов: задача и данные

Перед ассистентом стоит фильтр: он читает запрос студента и открытый фрагмент документа и отвечает
одной строкой, `PASS` или `BLOCK <категория>`. Пять категорий блокировки: обход проверки, выдуманные
источники, выдуманные данные, подгонка, работа за студента.

Задача узкая и с прямым ответом, поэтому метрики точные и без судьи: accuracy, полнота блокировок,
ложные блокировки, точность категории. Самое интересное — ловушки: легитимные просьбы, похожие
на нарушения, и ситуации, где ответ зависит от документа.

In [ ]:
import sys
sys.path.insert(0, "../..")

from src import data, infer
from src import filter as F

from collections import Counter

train, dev, test = F.load("train"), F.load("dev"), F.load("test")
for name, ds in (("train", train), ("dev", dev), ("test", test)):
    print(f"{name:6} {len(ds):4}  {dict(Counter(ds['label']))}  ловушек {sum(n == 'ловушка' for n in ds['note'])}")
print("категории:", dict(Counter(c for c in train["category"] if c)))

## Как выглядит строка

Тот же формат TRL, что и у ассистента: `prompt`, `chosen`, `rejected`. Ответ короткий, у блокировки
вторая строка — фиксированная фраза категории.

In [ ]:
for row in (test[0], next(r for r in test if r["label"] == "PASS" and r["document"])):
    print("═" * 78, row["id"], "·", row["label"], row["category"] or "")
    print(row["prompt"][1]["content"])
    print("\nЭТАЛОН:", row["chosen"][0]["content"].replace("\n", " | "))
    print("ПЛОХОЙ: ", row["rejected"][0]["content"].replace("\n", " | "))

## Решает документ

Одна и та же просьба перефразировать: свой черновик — PASS, чужая цитата — BLOCK.

In [ ]:
rows = list(train) + list(dev) + list(test)
for r in rows:
    if "ерефразируй" in r["request"] and r["document"] in ("own-colloquial", "quote-textbook", "own-conclusion"):
        print(f"{r['label']:5} {r['category'] or '':22} doc={r['document']:16} {r['request']}")

## Ловушки

Легитимные просьбы, которые звучат как нарушение. Именно на них база ставит лишние BLOCK, и именно
здесь методы обучения различаются сильнее всего.

In [ ]:
for r in [r for r in test if r["note"] == "ловушка"][:10]:
    print(f"  {r['request']}" + (f"   [doc: {r['document']}]" if r["document"] else ""))